# FIAP TECH CHALLENGE 1 - GRUPO SSP

## 2. ANÁLISE DOS DADOS

### 2.1. Estabelecendo a variável `BASE_PATH`

In [ ]:
import pandas as pd
from pathlib import Path

# Considerando que o Notebook está sendo rodado em /src/notebooks/
BASE_PATH = Path.cwd().resolve().parent.parent
print(f"📁 Base path: {BASE_PATH.absolute()}")

### 2.2. Gerando um Dataframe único com todos os arquivos parquet concatenados

In [ ]:
# Carregar todos os arquivos parquet do diretório
parquet_dir = BASE_PATH / "data" / "raw" / "VIOLBR24.parquet"

# Lista todos os arquivos .parquet no diretório
parquet_files = list(parquet_dir.glob("*.parquet"))
print(f"📊 Encontrados {len(parquet_files)} arquivos parquet")

# Carrega cada arquivo parquet e combina em um único dataframe
dfs = []
for parquet_file in parquet_files:
    df = pd.read_parquet(parquet_file)
    dfs.append(df)
    # print(f"✓ Carregado: {parquet_file.name} ({len(df)} linhas)")

# Concatena todos os dataframes
df = pd.concat(dfs, ignore_index=True)
print(f"\n✅ DataFrame final criado com {len(df)} linhas e {len(df.columns)} colunas")

### 2.3. Aplicando filtros para delimitar escopo

Filtros Aplicados:
- Sexo Feminino
- Lesões que não sejam autoprovacadas

In [ ]:
filter = (
    (df['CS_SEXO'] == 'F') &
    (df['LES_AUTOP'] == '2')
)
filtered_df = df[filter].copy()


### 2.4. Analisando o dataframe resultando `filtered_df`

In [ ]:
# Visualização inicial do dataframe
print("📋 Informações do DataFrame:")
print(f"Shape: {filtered_df.shape}")
print(f"\nColunas ({len(filtered_df.columns)}):")
print(filtered_df.columns.tolist())
print(f"\nTipos de dados:")
print(filtered_df.dtypes)
print(f"\nPrimeiras linhas:")
filtered_df.head()

### 2.4. Utilização de campos relevantes

Há cerca de 160 campos no dataset original vindo do DATASUS, dentre os quais cerca de 89 estão documentados no dicionário de dados mais atual. Diante desse impasse, criamos um dicionário de dados próprio e preenchemos as informações de cada coluna e adicionamos o campo `relevante`. Caso verdadeiro, o campo será usado neste estudo. Essa estratégia foi usada pois há muitos campos redundantes etc.

In [ ]:
import json

# Carregar o dicionário de campos
dicionario_path = BASE_PATH / "src" / "data" / "dicionario_campos_sinan.json"
with open(dicionario_path, 'r', encoding='utf-8') as f:
    dicionario = json.load(f)

# Filtrar campos com relevante = True
campos_relevantes = [
    campo for campo, info in dicionario.items() 
    if info.get('relevante', False) == True
]

# Filtrar apenas as colunas que existem no dataframe
colunas_existentes = [col for col in campos_relevantes if col in filtered_df.columns]
colunas_nao_existentes = [col for col in campos_relevantes if col not in filtered_df.columns]

if colunas_nao_existentes:
    print(f"\n⚠️  Aviso: {len(colunas_nao_existentes)} campos do dicionário não existem no dataframe:")
    print(colunas_nao_existentes)


# Criar subconjunto do dataframe com apenas as colunas relevantes
df_relevante = filtered_df[colunas_existentes].copy()

print(f"\n✅ DataFrame com campos relevantes criado:")
print(f"   Shape: {df_relevante.shape}")

### 2.5. Utilização de campos preenchidos

Analisando o dataset, percebemos que há muitos campos que estão praticamente sem dado algum. Fizemos uma análise para verificar o percentual de não preenchimento do campo, considerando valores como NaN ou `' '` no em cada registro do dataframe. Ao final, o novo dataframe foi salvo no parquet `df_cleaned.parquet`

In [ ]:
# Valores null/NaN
valores_null = df_relevante.isna().sum()

# Strings vazias ou com apenas espaços
valores_string_vazia = df_relevante.apply(
    lambda col: (col.astype(str).str.strip() == '').sum()
)

# print(f'valores_null: {valores_null}')
# print(f'valores_string_vazia: {valores_string_vazia}')

# dataframes com valores null e valores string vazia
# valores_vazio_df = pd.concat([valores_null, valores_string_vazia], axis=1, keys=['null', 'string_vazia'])
# print(valores_vazio_df)

# printar somente LESAO_ESPE (linhas = campos, colunas = tipo de vazio)
# print(valores_vazio_df.loc['LESAO_ESPE'])

# Total combinado
valores_vazios = valores_null + valores_string_vazia

percentual_vazios = (valores_vazios / len(df_relevante)) * 100
# formatar percentual_vazios em 2 decimais
percentual_vazios = percentual_vazios.round(5)
# ordenar percentual_vazios de forma decrescente
percentual_vazios = percentual_vazios.sort_values(ascending=False)

# print(f'valores_vazios (total): {valores_vazios}')

print(f'percentual_vazios: {percentual_vazios}')

# criar dataframe em que as colunas são as que tem menos de 10% de vazios
cleaned_df = df_relevante.loc[:, percentual_vazios < 10]
# print(cleaned_df.shape)
# print(cleaned_df.columns)
# print(cleaned_df.head())

# recalcular percentual de vazios
ev2 = cleaned_df.apply(
    lambda col: (col.astype(str).str.strip() == '').sum()
)
pv2 = (ev2 / len(cleaned_df)) * 100
pv2 = pv2.round(2)
pv2 = pv2.sort_values(ascending=False)

print(f'percentual_vazios: {pv2}')

# print quantidade de colunas do dataframe atual
print(f'quantidade de colunas do dataframe atual: {len(cleaned_df.columns)}')

# salvar em data/processed/df_cleaned.parquet
cleaned_df.to_parquet(BASE_PATH / "data" / "processed" / "df_cleaned.parquet")


percentual_vazios: TPUNINOT      100.00000
ENC_ESPEC      99.99864
LESAO_ESPE     99.99796
LESAO_CORP     99.99185
LESAO_NAT      99.99049
                ...    
SG_UF           0.01155
ID_AGRAVO       0.00000
CS_SEXO         0.00000
LES_AUTOP       0.00000
DT_OCOR         0.00000
Length: 79, dtype: float64


### 2.6. Distribuição de Valores

Nesta etapa analisamos a distribuição dos valores de cada um dos campos remanescentes a fim de verificar quais deles são elegíveis para serem utilizados em nosso estudo.

LEGENDA:
Para a maioria das respostas:
1. Sim
2. Não
9. Ignorado 

Orientação Sexual: 
1. Heterossexual
2. Homossexual
(gay/lésbica)
3. Bissexual
8. Não se aplica
9. Ignorado

Autor_Sexo:
1. Masculino
2. Feminino
3. Ambos os sexos
9 .Ignorado

Identidade de Gênero:
1. Travesti
2. Transexual Mulher
3. Transexual Homem
8. Não se aplica
9. Ignorado

Motivo da Violência:
01. Sexismo
02. Homofobia/Lesbofobia
Bifobia/Transfobia
03. Racismo
04. Intolerância religiosa
05. Xenofobia
06. Conflito geracional
07. Situação de rua
08. Deficiência
09. Outros
88. Não se aplica
99. Ignorado

Escolaridade:
1. 1ª a 4ª série incompleta do
EF
2. 4ª série completa do EF (
antigo 1° grau)
3. 5ª à 8ª série incompleta do
EF (antigo ginásio ou 1°
grau)
4. Ensino fundamental
completo (antigo ginásio ou
1° grau)
5. Ensino médio incompleto
(antigo colegial ou 2° grau)
6. Ensino médio completo
(antigo colegial ou 2° grau)
7. Educação superior
incompleta
8. Educação superior
completa
9. Ignorado
10. Não se aplica


Raça: 
1- branca
2- preta
3- amarela
4- parda
5- indígena
9 Ignorado

Deficiências:
1. Sim
2. Não
8. Não se aplica
9. Ignorado


In [ ]:
# Analisar a distribuição de valores de cada coluna
for col in cleaned_df.columns:
    print(f'Distribuição de valores da coluna {col}:')
    print(cleaned_df[col].value_counts())
    print('\n')


In [ ]:
cleaned_df.loc['DT_OCOR_CONV'] = pd.to_datetime(cleaned_df['DT_OCOR'].astype(str), format='%Y%m%d', errors='coerce')


cleaned_df.loc['DIA_SEMANA_OCOR'] = cleaned_df['DT_OCOR_CONV'].dt.dayofweek.map({0:'SEG',1:'TER',2:'QUA',3:'QUI',4:'SEX',5:'SAB',6:'DOM'})
cleaned_df.loc['MES_OCOR'] = cleaned_df['DT_OCOR_CONV'].dt.month.map({1:'JAN',2:'FEV',3:'MAR',4:'ABR',5:'MAI',6:'JUN',7:'JUL',8:'AGO',9:'SET',10:'OUT',11:'NOV',12:'DEZ'})

cleaned_df[['DT_OCOR','DT_OCOR_CONV','DIA_SEMANA_OCOR','MES_OCOR']].head()

# mostrar a distribuição de valores da coluna MES_OCOR
print(cleaned_df['MES_OCOR'].value_counts())

# mostrar a distribuição de valores da coluna DIA_SEMANA_OCOR
print(cleaned_df['DIA_SEMANA_OCOR'].value_counts())

# 

## Histogramas e Box-plot

In [ ]:
pip install numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TOP_N = 20

num_cols = cleaned_df.select_dtypes(include=np.number).columns
cat_cols = cleaned_df.columns.difference(num_cols)

def plot_col(c):
    s = cleaned_df[c].dropna()
    if s.empty:
        return

    sn = pd.to_numeric(s, errors="coerce").dropna()

    if not sn.empty:  # numérica
        fig, ax = plt.subplots(1, 2, figsize=(10, 3))
        ax[0].hist(sn, bins=30)
        ax[0].set_title(f"Hist: {c}")
        ax[1].boxplot(sn, vert=False)
        ax[1].set_title(f"Box: {c}")
        plt.tight_layout()
        plt.show()
        print(c, sn.describe())
    else:  # categórica
        vc = s.astype(str).str.strip().replace({"": np.nan}).value_counts(dropna=False).head(TOP_N)
        vc = vc.sort_values(ascending=True)  # barh: maior em cima
        fig, ax = plt.subplots(figsize=(10, 0.35*len(vc)+1))
        bars = ax.barh(vc.index.astype(str), vc.values)
        ax.set_title(f"{c} (top {TOP_N})")
        ax.bar_label(bars, padding=3, fontsize=9)
        plt.tight_layout()
        plt.show()
        print(c, s.describe())

# roda pra todas (ou use cleaned_df.columns[:10] pra testar)
for c in cleaned_df.columns:
    plot_col(c)